# Rolling Loader Low-Level Probe

Use this notebook to isolate the low-level ClickHouse calls behind the rolling training loader. Start by running the setup cell, then run the Arrow/Polars first-window query cell.

In [ ]:
from __future__ import annotations

import datetime as dt
import sys
import time
from pathlib import Path
from urllib.parse import urlparse

try:
    import clickhouse_connect
    import polars as pl
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "This notebook needs clickhouse-connect, polars, and pyarrow in the active Python environment. "
        "Install them in the workstation environment, then rerun this cell."
    ) from exc

cwd = Path.cwd().resolve()
REPO_ROOT = next(path for path in (cwd, *cwd.parents) if (path / "research" / "mlops" / "rolling_loader").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from research.mlops.clickhouse import (
    ClickHouseHttpClient,
    default_clickhouse_password,
    default_clickhouse_url,
    default_clickhouse_user,
    discover_clickhouse_env_files,
    quote_ident,
    sql_string,
)
from research.mlops.clickhouse_events import PersistentClickHouseBytesClient
from research.mlops.env import load_env_files
from research.mlops.rolling_loader.config import RollingLoaderConfig
from research.mlops.rolling_loader.loader import RollingContextLoader
from research.mlops.rolling_loader.sources import (
    ClickHouseExternalContextConfig,
    ClickHouseExternalContextSource,
    ClickHouseReplayConfig,
    ClickHouseRollingSource,
)

START_UTC = "2026-06-01T00:00:00+00:00"
REPLAY_WINDOW_US = 200_000
EVENT_PREFETCH_ROWS = 8_192
EVENT_LOOKAHEAD_DAYS = 7
TICKER_LIMIT = 0

load_env_files(discover_clickhouse_env_files())

clickhouse_url = default_clickhouse_url()
clickhouse_user = default_clickhouse_user()
clickhouse_password = default_clickhouse_password()

parsed = urlparse(clickhouse_url)
secure = parsed.scheme == "https"
client = clickhouse_connect.get_client(
    host=parsed.hostname or "localhost",
    port=parsed.port or (8443 if secure else 8123),
    username=clickhouse_user,
    password=clickhouse_password,
    secure=secure,
)

text_client = ClickHouseHttpClient(clickhouse_url, clickhouse_user, clickhouse_password)
bytes_client = PersistentClickHouseBytesClient(clickhouse_url, clickhouse_user, clickhouse_password)

replay_config = ClickHouseReplayConfig(database="market_sip_compact", max_threads=8, max_memory_usage="80G")
context_config = ClickHouseExternalContextConfig(database=replay_config.database, sec_context_database=replay_config.database)
loader_config = RollingLoaderConfig()

source = ClickHouseRollingSource(config=replay_config, text_client=text_client, bytes_client=bytes_client)
context_source = ClickHouseExternalContextSource(config=context_config, text_client=text_client)
loader = RollingContextLoader(config=loader_config)

start_timestamp_us = int(dt.datetime.fromisoformat(START_UTC).timestamp() * 1_000_000)
start_date = dt.datetime.fromtimestamp(start_timestamp_us / 1_000_000, tz=dt.timezone.utc).date()
end_date = start_date + dt.timedelta(days=EVENT_LOOKAHEAD_DAYS)

print({
    "repo_root": str(REPO_ROOT),
    "clickhouse_url": clickhouse_url,
    "database": replay_config.database,
    "events_table": replay_config.events_table,
    "index_table": replay_config.index_table,
    "start_utc": START_UTC,
    "start_timestamp_us": start_timestamp_us,
    "replay_window_us": REPLAY_WINDOW_US,
    "event_prefetch_rows": EVENT_PREFETCH_ROWS,
    "warmup_events_per_ticker": loader_config.warmup_events_per_ticker,
})

In [ ]:
loader.cache_summary()

In [ ]:
events_table = f"{quote_ident(replay_config.database)}.{quote_ident(replay_config.events_table)}"
start_date_sql = f"toDate({sql_string(start_date.isoformat())})"
end_date_sql = f"toDate({sql_string(end_date.isoformat())})"

first_window_query = f"""
WITH
    toUInt64({start_timestamp_us}) AS request_start_us,
    toUInt64({REPLAY_WINDOW_US}) AS request_window_us,
    (
        SELECT minOrNull(sip_timestamp_us)
        FROM {events_table}
        PREWHERE event_date >= {start_date_sql}
          AND event_date < {end_date_sql}
        WHERE sip_timestamp_us > request_start_us
    ) AS window_start_us
SELECT
    ticker,
    ordinal,
    event_type,
    sip_timestamp_us,
    price_primary_int,
    price_secondary_int,
    size_primary,
    size_secondary,
    exchange_primary,
    exchange_secondary,
    event_flags,
    conditions_packed
FROM {events_table}
PREWHERE event_date >= {start_date_sql}
  AND event_date <= toDate(fromUnixTimestamp64Micro(window_start_us + request_window_us))
WHERE sip_timestamp_us >= window_start_us
  AND sip_timestamp_us <= window_start_us + request_window_us
ORDER BY sip_timestamp_us, ticker, ordinal
SETTINGS max_threads = {int(replay_config.max_threads)}, max_memory_usage = {source._memory_bytes(replay_config.max_memory_usage)}
"""

print(first_window_query)
t0 = time.perf_counter()
arrow_table = client.query_arrow(first_window_query)
query_seconds = time.perf_counter() - t0

# Read the Arrow table straight into a Polars DataFrame.
df = pl.from_arrow(arrow_table)

print({
    "query_seconds": round(query_seconds, 6),
    "rows": df.height,
    "columns": df.width,
    "tickers": df.select(pl.col("ticker").n_unique()).item() if df.height else 0,
    "min_sip_timestamp_us": df.select(pl.col("sip_timestamp_us").min()).item() if df.height else None,
    "max_sip_timestamp_us": df.select(pl.col("sip_timestamp_us").max()).item() if df.height else None,
})
df.head(20)

In [ ]:
df.shape

In [ ]:
# Table-aligned fetch: first get all tickers in the selected month, then get the first 10 events per ticker.
# This uses the shape the current table is good at: ticker IN (...) and ordinal BETWEEN per ticker.
FIRST_EVENTS_PER_TICKER = 10

month_start = dt.date(start_date.year, start_date.month, 1)
next_month = (month_start.replace(day=28) + dt.timedelta(days=4)).replace(day=1)
month_start_sql = f"toDate({sql_string(month_start.isoformat())})"
next_month_sql = f"toDate({sql_string(next_month.isoformat())})"

month_tickers_query = f"""
SELECT
    ticker,
    min(ordinal) AS first_ordinal,
    max(ordinal) AS last_ordinal,
    count() AS month_events
FROM {events_table}
PREWHERE event_date >= {month_start_sql}
  AND event_date < {next_month_sql}
GROUP BY ticker
ORDER BY ticker
SETTINGS max_threads = {int(replay_config.max_threads)}, max_memory_usage = {source._memory_bytes(replay_config.max_memory_usage)}
"""

print(month_tickers_query)
t0 = time.perf_counter()
arrow_table_month_tickers = client.query_arrow(month_tickers_query)
month_tickers_query_seconds = time.perf_counter() - t0
df_month_tickers = pl.from_arrow(arrow_table_month_tickers)

print({
    "month_start": month_start.isoformat(),
    "next_month": next_month.isoformat(),
    "query_seconds": round(month_tickers_query_seconds, 6),
    "tickers": df_month_tickers.height,
    "month_events": int(df_month_tickers.get_column("month_events").sum()) if df_month_tickers.height else 0,
})
display(df_month_tickers.head(20))

if df_month_tickers.height == 0:
    raise RuntimeError(f"No tickers found for month_start={month_start.isoformat()}.")

request_tickers = [str(ticker) for ticker in df_month_tickers.get_column("ticker").to_list()]
request_start_ordinals = [int(value) for value in df_month_tickers.get_column("first_ordinal").to_list()]
request_end_ordinals = [int(value) + FIRST_EVENTS_PER_TICKER - 1 for value in df_month_tickers.get_column("first_ordinal").to_list()]

ticker_sql = "[" + ", ".join(sql_string(ticker) for ticker in request_tickers) + "]"
start_ordinal_sql = "[" + ", ".join(str(value) for value in request_start_ordinals) + "]"
end_ordinal_sql = "[" + ", ".join(str(value) for value in request_end_ordinals) + "]"

ordinal_range_query = f"""
WITH
    {ticker_sql} AS request_tickers,
    {start_ordinal_sql} AS request_start_ordinals,
    {end_ordinal_sql} AS request_end_ordinals
SELECT
    toUInt32(indexOf(request_tickers, ticker) - 1) AS span_id,
    ticker,
    ordinal,
    event_type,
    sip_timestamp_us,
    price_primary_int,
    price_secondary_int,
    size_primary,
    size_secondary,
    exchange_primary,
    exchange_secondary,
    event_flags,
    conditions_packed
FROM {events_table}
PREWHERE event_date >= {month_start_sql}
  AND event_date < {next_month_sql}
  AND ticker IN request_tickers
WHERE ordinal >= arrayElement(request_start_ordinals, indexOf(request_tickers, ticker))
  AND ordinal <= arrayElement(request_end_ordinals, indexOf(request_tickers, ticker))
ORDER BY ticker, ordinal
SETTINGS max_threads = {int(replay_config.max_threads)}, max_memory_usage = {source._memory_bytes(replay_config.max_memory_usage)}
"""

print(ordinal_range_query)
t0 = time.perf_counter()
arrow_table_ordinal = client.query_arrow(ordinal_range_query)
ordinal_query_seconds = time.perf_counter() - t0

# Polars DataFrame for the table-aligned ordinal-range path.
df_ordinal = pl.from_arrow(arrow_table_ordinal)

print({
    "query_seconds": round(ordinal_query_seconds, 6),
    "requested_tickers": len(request_tickers),
    "first_events_per_ticker": FIRST_EVENTS_PER_TICKER,
    "rows": df_ordinal.height,
    "columns": df_ordinal.width,
    "min_sip_timestamp_us": df_ordinal.select(pl.col("sip_timestamp_us").min()).item() if df_ordinal.height else None,
    "max_sip_timestamp_us": df_ordinal.select(pl.col("sip_timestamp_us").max()).item() if df_ordinal.height else None,
})
df_ordinal.head(20)

In [ ]:
# Optional cleanup when you are done with live probes.
source.close()
client.close()